# Notebook 10 — Model Audit and Error Analysis

## Goal
Inspect whether the trained XGBoost models are trustworthy before using their predictions
in the RAG explanation layer. A model with inexplicable errors should not be used.

---

## What can go wrong
- Models may perform well on average but fail catastrophically on crisis periods
- Feature importance may show unexpected features dominating (data leakage signal)
- Errors may be systematically biased in one direction
- Residuals may be autocorrelated (model is missing a pattern)

---

## What you must inspect
- Actual vs. predicted scatter: is there a diagonal pattern?
- Error over time: are errors clustered in a specific period?
- Worst 10 forecast errors: are they all in COVID? (acceptable) or spread out? (concerning)
- Feature importance: does it make economic sense?

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

MODEL_READY_PATH = DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet'
PREDS_DIR = DRIVE_ROOT / 'outputs' / 'predictions'
FIGS_DIR = DRIVE_ROOT / 'outputs' / 'figures'
TABLES_DIR = DRIVE_ROOT / 'outputs' / 'tables'

ap_path = DRIVE_ROOT / 'approvals' / '09_xgboost_models_approved.json'
if not ap_path.exists():
    raise FileNotFoundError('STOP: Notebook 09 not approved.')
with open(ap_path) as f:
    ap = json.load(f)
assert ap['approved'], 'NB 09 not approved.'

with open(REPO_DIR / 'configs' / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

MACRO_FEATURES = model_cfg['macro_features']
NEWS_FEATURES = model_cfg['news_features']
ALL_FEATURES = MACRO_FEATURES + NEWS_FEATURES
TARGET = model_cfg['target_column']
TEST_START = pd.Timestamp(model_cfg['splits']['test_start'])

model_df = pd.read_parquet(MODEL_READY_PATH)
model_df['forecast_date'] = pd.to_datetime(model_df['forecast_date'])
model_df = model_df.dropna(subset=[TARGET]).sort_values('forecast_date').reset_index(drop=True)
test_df = model_df[model_df['forecast_date'] >= TEST_START]

# Load models
xgb_macro = joblib.load(DRIVE_ROOT / 'artifacts' / 'models' / 'xgboost_macro_only.joblib')
xgb_mn = joblib.load(DRIVE_ROOT / 'artifacts' / 'models' / 'xgboost_macro_news.joblib')

# Load predictions
macro_pred_df = pd.read_parquet(PREDS_DIR / 'predictions_macro_only.parquet')
mn_pred_df = pd.read_parquet(PREDS_DIR / 'predictions_macro_news.parquet')

macro_pred_df['forecast_date'] = pd.to_datetime(macro_pred_df['forecast_date'])
mn_pred_df['forecast_date'] = pd.to_datetime(mn_pred_df['forecast_date'])
macro_pred_df['error'] = macro_pred_df['actual'] - macro_pred_df['prediction']
mn_pred_df['error'] = mn_pred_df['actual'] - mn_pred_df['prediction']

print(f'Test predictions loaded: {len(macro_pred_df)} macro-only, {len(mn_pred_df)} macro+news')

## Actual vs. Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred_df, name in [
    (axes[0], macro_pred_df, 'Macro-Only'),
    (axes[1], mn_pred_df, 'Macro+News'),
]:
    ax.scatter(pred_df['actual'], pred_df['prediction'], alpha=0.6, s=40)
    lim = max(abs(pred_df['actual'].max()), abs(pred_df['actual'].min())) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1, label='Perfect forecast')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('Actual (K)')
    ax.set_ylabel('Predicted (K)')
    ax.set_title(f'{name}: Actual vs. Predicted')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGS_DIR / 'actual_vs_predicted.png'), dpi=100)
plt.show()
print('Saved: actual_vs_predicted.png')

## Errors over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, pred_df, name, color in [
    (axes[0], macro_pred_df, 'Macro-Only', 'steelblue'),
    (axes[1], mn_pred_df, 'Macro+News', 'darkorange'),
]:
    bar_colors = ['red' if e < 0 else color for e in pred_df['error']]
    ax.bar(pred_df['forecast_date'], pred_df['error'], color=bar_colors, alpha=0.7, width=20)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{name}: Forecast Error Over Time (actual − predicted)')
    ax.set_ylabel('Error (K)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGS_DIR / 'error_over_time.png'), dpi=100)
plt.show()
print('Saved: error_over_time.png')

## Residual distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred_df, name in [
    (axes[0], macro_pred_df, 'Macro-Only'),
    (axes[1], mn_pred_df, 'Macro+News'),
]:
    ax.hist(pred_df['error'], bins=20, edgecolor='black', alpha=0.7)
    ax.axvline(pred_df['error'].mean(), color='red', linestyle='--', label=f'Mean={pred_df["error"].mean():.1f}K')
    ax.set_title(f'{name}: Residual Distribution')
    ax.set_xlabel('Error (K)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGS_DIR / 'residual_distribution.png'), dpi=100)
plt.show()

## Worst 10 and Best 10 forecast errors

In [ ]:
# Use macro+news model as primary
worst = mn_pred_df.reindex(mn_pred_df['error'].abs().nlargest(10).index)
best = mn_pred_df.reindex(mn_pred_df['error'].abs().nsmallest(10).index)

print('=== WORST 10 FORECAST ERRORS (Macro+News) ===')
print(worst[['forecast_date', 'actual', 'prediction', 'error']].to_string(index=False))
print()
print('=== BEST 10 FORECASTS (smallest |error|) ===')
print(best[['forecast_date', 'actual', 'prediction', 'error']].to_string(index=False))

worst_path = TABLES_DIR / 'worst_forecast_errors.csv'
worst.to_csv(worst_path, index=False)
print(f'\nWorst errors saved: {worst_path}')

## Error by year (crisis vs. non-crisis)

In [ ]:
mn_pred_df['year'] = mn_pred_df['forecast_date'].dt.year
yearly = mn_pred_df.groupby('year').agg(
    MAE=('error', lambda x: x.abs().mean()),
    mean_error=('error', 'mean'),
    n=('error', 'count')
).reset_index()

print('=== MAE by Year (Macro+News) ===')
print(yearly.to_string(index=False))

# Crisis vs non-crisis
crisis_months = mn_pred_df[mn_pred_df['forecast_date'].dt.year.isin([2020])]
non_crisis = mn_pred_df[~mn_pred_df['forecast_date'].dt.year.isin([2020])]

print(f'\nCrisis (2020) MAE:     {crisis_months["error"].abs().mean():.1f}K (n={len(crisis_months)})')
print(f'Non-crisis MAE:       {non_crisis["error"].abs().mean():.1f}K (n={len(non_crisis)})')

## Feature importance

In [ ]:
available_all = [f for f in ALL_FEATURES if f in model_df.columns]
importances = pd.Series(
    xgb_mn.feature_importances_,
    index=available_all
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['darkorange' if f in NEWS_FEATURES else 'steelblue' for f in importances.index]
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Importance — XGBoost Macro+News\n(orange = news features, blue = macro features)')
ax.set_xlabel('Importance score')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGS_DIR / 'feature_importance.png'), dpi=100)
plt.show()

print('\nFeature importances (sorted):')
print(importances.sort_values(ascending=False).to_string())
print('\nSaved: feature_importance.png')

## Sanity check: does feature importance make economic sense?

**Expected patterns:**
- `payems_lag1` should have high importance (momentum effect)
- `unrate_lag1` and `icsa_lag1` should contribute (leading indicators)
- News features should rank below macro features unless news adds unique signal
- If a news feature has suspiciously high importance, check for data leakage

Inspect the chart above and confirm the pattern makes economic sense.

In [ ]:
top_feature = importances.sort_values(ascending=False).index[0]
print(f'Most important feature: {top_feature}')

if top_feature in NEWS_FEATURES:
    print(f'WARNING: A news feature ({top_feature}) is the most important. Verify this is not leakage.')
else:
    print('OK: A macro feature is most important — expected.')

# Check for suspicious patterns
if 'payems_lag1' in importances:
    payems_rank = list(importances.sort_values(ascending=False).index).index('payems_lag1') + 1
    print(f'payems_lag1 rank: {payems_rank} of {len(importances)}')

## Note: No approval required for this notebook

This is an audit notebook. Its outputs inform your assessment of model trustworthiness.
Findings here should be documented in `reports/methods.md` and the paper's error analysis section.

Proceed to `11_build_rag_evidence_retrieval.ipynb` when satisfied.